# Forest Disturbance Detection - Modular Training Pipeline

Clean demonstration of the modularized training workflow.
All code is now organized in the `disturbance_detection` package.

**Sections:**
1. Setup & Configuration
2. Data Loading & Preparation
3. Model Training
4. Evaluation & Visualization

### Setup and Configuration

In [ ]:
import pandas as pd
import torch
import os

# Import from our modular package
from disturbance_detection import (
    Config, set_seed,
   print_split_balances, #make_or_load_uid_splits,  prepare_data, 
    UNet_1D_W3, FocalLoss,
    safe_to_device,
    final_eval_with_val_threshold,
    count_supervised_positives,
    plot_history, plot_precision_recall_curve, 
    plot_f1_vs_threshold, print_classification_report,
    get_model,
    TemporalCNN,
    UNet_1D_W30,
    UNet_1D_W5to7,
    UNet_1D_W5to7_MultiLevel_Attention,
    CaptumEvaluator,
    evaluate_position_wise_metrics,
    load_best_model_and_evaluate_position_wise_metrics,
    TemporalMultiScaleUNet
)

from disturbance_detection.preprocessing.data_preprocessing import prepare_data, make_or_load_uid_splits
print("Direct import works!")

print("All imports successful!")

In [ ]:
# Create configuration
config = Config()

# Set random seed for reproducibility
set_seed(config.seed)

# Check device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
config.target_mode = "last"
config.window_size = 5

config.focal_alpha = 0.3 #0.8
config.focal_gamma = 1.9 #2.0
config.features_mode = "bands_indices"
config.num_epochs = 30
config.scheduler_patience = 5
config.num_workers = 8


# Display key configuration
print(f"\nConfiguration:")
print(f"  Features: {config.features_mode} ({config.get_num_features()} features)")
print(f"  Window size: {config.window_size}")
print(f"  Batch size: {config.batch_size}")
print(f"  Epochs: {config.num_epochs}")
print(f"  Learning rate: {config.learning_rate}")
print(f"  Focal loss: alpha={config.focal_alpha}, gamma={config.focal_gamma}")

### Dataloading and Preparation

In [ ]:
# Load dataset
df = pd.read_csv(config.csv_path, index_col=0, sep=config.csv_separator)
print(f"Loaded {len(df)} samples")

# Create disturbance labels
df['class'] = df['class_level1'].apply(lambda x: 1 if x == 'disturbance' else 0)

# Filter doubtful samples (optional - from original notebook)
df = df[~((df['class_level1'] == 'treed') & (df['NBR'] < 0.5))]
df = df[~((df['class_level1'] == 'treed') & (df['NDVI'] < 0.5))]
df = df[~((df['class_level1'] == 'non-treed') & (df['NBR'] < 0.5))]
df = df[~((df['class_level1'] == 'non-treed') & (df['NDVI'] < 0.5))]
#df = df[~((df['class_level1'] == 'disturbance') & (df['NBR'] > 0.5))]
#df = df[~((df['class_level1'] == 'disturbance') & (df['NDVI'] > 0.5))]

print(f"After filtering: {len(df)} samples")
print(f"\nClass distribution:")
print(df['class_level1'].value_counts())

In [ ]:
# Prepare train/val/test splits and dataloaders
train_loader, val_loader, test_loader, n_features, features = prepare_data(df, config)

print(f"\nData preparation complete!")
print(f"Features used: {features}")
print(f"Number of features: {n_features}")
print(f"\nDataLoader sizes:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

In [ ]:
# Check class balance at supervised position
tr_pos, tr_tot, tr_rate = count_supervised_positives(train_loader, config.target_mode, device)
va_pos, va_tot, va_rate = count_supervised_positives(val_loader, config.target_mode, device)
te_pos, te_tot, te_rate = count_supervised_positives(test_loader, config.target_mode, device)

print(f"Class balance at target position ('{config.target_mode}'):")
print(f"  Train: {tr_pos}/{tr_tot} ({100*tr_rate:.3f}% positive)")
print(f"  Val:   {va_pos}/{va_tot} ({100*va_rate:.3f}% positive)")
print(f"  Test:  {te_pos}/{te_tot} ({100*te_rate:.3f}% positive)")

In [ ]:
# Check label distribution at each position in the window
import numpy as np

# Collect all labels from training data
all_labels = []
for x, y in train_loader:
    all_labels.append(y.numpy())

all_labels = np.concatenate(all_labels, axis=0)  # Shape: (N_samples, window_size)

print(f"Label distribution by position (window_size={config.window_size}):")
print("-" * 50)
for i in range(config.window_size):
    pos_name = "last" if i == config.window_size - 1 else ("center" if i == config.window_size // 2 else f"{i}")
    mean_val = all_labels[:, i].mean()
    count = int(all_labels[:, i].sum())
    total = len(all_labels)
    print(f"Position {i:2d} ({pos_name:>6s}): {mean_val:.4f} ({count}/{total} positive)")

In [ ]:
# Investigation 1: Check if disturbances persist across consecutive years
print("=== Checking label persistence ===")

# For each unique pixel (uniqueid), check how often disturbances persist
persistence_check = []
for uid, group in df.groupby('uniqueid'):
    group = group.sort_values('year')
    labels = group['class'].values
    
    # Find transitions from 0 -->1 (disturbance starts)
    for i in range(len(labels)-1):
        if labels[i] == 1:  # if disturbed
            # check if next year is also disturbed
            persistence_check.append(labels[i+1])

if len(persistence_check) > 0:
    persistence_rate = np.mean(persistence_check)
    print(f"When a pixel is disturbed, it stays disturbed next year: {persistence_rate:.2%} of the time")
    print(f"(Based on {len(persistence_check)} disturbance-years)")
else:
    print("No consecutive disturbances found")

print("\n" + "="*60)
print("=== Checking temporal trend (disturbance rate over years) ===\n")

# Investigation 2: Check if disturbance rate changes over time
yearly_stats = df.groupby('year').agg({
    'class': ['sum', 'count', 'mean']
}).round(4)

yearly_stats.columns = ['Disturbed', 'Total', 'Rate']
print(yearly_stats)

print(f"\nFirst year rate: {yearly_stats['Rate'].iloc[0]:.4f}")
print(f"Last year rate: {yearly_stats['Rate'].iloc[-1]:.4f}")
print(f"Increase: {(yearly_stats['Rate'].iloc[-1] / yearly_stats['Rate'].iloc[0] - 1)*100:.1f}%")

### Model Training

In [ ]:
# Initialize model

#Select model
#config.model_name = "TemporalCNN"
#model = get_model(config)


#or train UNet
config.model_name = "UNet_1D_W5to7"
#config.model_name = "UNet_1D_W30"
#config.model_name = "UNet_1D_W3"
#config.model_name = "UNet_1D_W5to7_MultiLevel_Attention"
#config.model_name = "TemporalMultiScaleUNet"
config.dropout_rate = 0.4
model = get_model(config).to(device)


# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model created: {config.model_name}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

In [ ]:
import torch.nn as nn

# Create optimizer
'''optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
    betas=config.betas,
    eps=config.eps
)'''

config.learning_rate = 0.001



optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config.learning_rate,
    betas=config.betas,
    eps=config.eps
)


# Create loss function
loss_fn = FocalLoss(
    alpha=config.focal_alpha,
    gamma=config.focal_gamma,
    reduction=config.loss_reduction
)


print("Optimizer: Adam")
print(f"  Learning rate: {config.learning_rate}")
print(f"  Weight decay: {config.weight_decay}")
print(f"\nLoss: FocalLoss")
print(f"  Alpha: {config.focal_alpha}")
print(f"  Gamma: {config.focal_gamma}")

In [ ]:
from disturbance_detection.training import train_full_supervision_with_selection

# Train with best model selection
print(f"\nStarting training for {config.num_epochs} epochs...")
print(f"Model: {config.model_name}")
print(f"Target position: {config.target_mode}")
print(f"Model selection by: {config.select_by}")
print(f"Window size: {config.window_size}")
print(f"Batch size: {config.batch_size}")
print("-" * 80)


config.norm_type = "bn"
config.dropout_rate = 0.4
config.batch_size = 32

history, best_info = train_full_supervision_with_selection(
    model, train_loader, val_loader, optimizer, device, loss_fn, config
)

print("-" * 80)
print(f"\nTraining complete!")
print(f"Best epoch: {best_info['best_epoch']}")
print(f"Best {best_info['select_by']}: {best_info['best_metric']:.4f}")
print(f"Checkpoint saved: {best_info['ckpt_path']}")

### Evaluation and Visualization

In [ ]:
# Import the function with a different name
from disturbance_detection.evaluations import plot_history as plot_training_history

# Create a mapping for the plot function
plot_data = {
    "train_loss": history["train_loss"],
    "val_loss": history["val_loss"],
    "train_f1": history["train_f1_all"],  # Use f1_all as the main F1
    "val_f1": history["val_f1_all"],      # Use f1_all as the main F1
    "val_auprc": history["val_auprc"]
}

# Then plot
plot_training_history(plot_data, title_suffix=f" (target={config.target_mode})")

In [ ]:
# Get position-wise metrics
import matplotlib.pyplot as plt

position_metrics = evaluate_position_wise_metrics(model, val_loader, device, config)

#Print position-wise metrics
for pos, metrics in enumerate(position_metrics):
    print(f"Position {pos}: F1={metrics['f1']:.3f}, Precision={metrics['precision']:.3f}, Recall={metrics['recall']:.3f}")


positions = [metrics['position'] for metrics in position_metrics]
f1_scores = [metrics['f1'] for metrics in position_metrics]

plt.figure(figsize=(6, 4))
plt.bar(positions, f1_scores, alpha=0.7)
plt.xlabel('Position in Sequence')
plt.ylabel('F1 Score')
plt.title('F1 Score by Position in Sequence')
#plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
#plt.savefig('position_wise_f1.png', dpi=150)
plt.show()

In [ ]:
# Load best model and evaluate
print("Evaluating best model on test set...")
print("=" * 80)

best_thr, test_metrics = final_eval_with_val_threshold(
    model, device, config.checkpoint_path,
    val_loader, test_loader, 
    target_mode=config.target_mode
)

print("=" * 80)

In [ ]:
from disturbance_detection.evaluations import collect_probs

# Collect test predictions
y_true_test, y_prob_test = collect_probs(model, test_loader, device, config.target_mode)

# Print detailed classification report
print_classification_report(
    y_true_test, y_prob_test, best_thr,
    target_names=["Undisturbed (0)", "Disturbed (1)"]
)

In [ ]:
# F1 score as a function of threshold
plot_f1_vs_threshold(y_true_test, y_prob_test, best_thr, 
                     "Test Set: F1 Score vs Threshold")

In [ ]:
from disturbance_detection.evaluations import plot_confusion_matrix

# Plot confusion matrix
plot_confusion_matrix(test_metrics['cm'], 
                     labels=["Undisturbed", "Disturbed"],
                     title="Test Set: Confusion Matrix")

In [ ]:
def quick_evaluation(model, test_loader, config, device, num_samples: int = 3):
    """Quick attribution evaluation - just run this function!"""
    
    evaluator = CaptumEvaluator(model, device, config)
    
    # Find positive samples
    positive_samples = []
    for batch_x, batch_y in test_loader:
        for i in range(batch_x.shape[0]):
            if batch_y[i, -1] == 1:  # Disturbance at last timestep
                positive_samples.append(batch_x[i:i+1])
                if len(positive_samples) >= num_samples:
                    break
        if len(positive_samples) >= num_samples:
            break
    
    if not positive_samples:
        print("No positive samples found!")
        return
    
    print(f"Analyzing {len(positive_samples)} disturbance samples...")
    
    # Analyze each sample
    for i, sample in enumerate(positive_samples):
        print(f"\n--- Sample {i+1} ---")
        
        # Get prediction - handle different output formats
        with torch.no_grad():
            logits = model(sample.to(device))
            
            # Handle different model output formats
            if len(logits.shape) > 1 and logits.shape[1] > 1:
                # Model outputs (batch_size, time_steps) - take target position
                if config.target_mode == "last":
                    target_logit = logits[0, -1]
                elif config.target_mode == "center":
                    center_idx = config.window_size // 2
                    target_logit = logits[0, center_idx]
                else:
                    target_logit = logits[0, -1]  # Default to last
            else:
                # Model outputs single value per sample
                target_logit = logits[0] if len(logits.shape) > 0 else logits
            
            pred = torch.sigmoid(target_logit)
            print(f"Model prediction: {pred.item():.3f}")
        
        # Get attributions
        ig_attr = evaluator.get_attribution(sample, 'integrated_gradients')
        sal_attr = evaluator.get_attribution(sample, 'saliency')
        
        # Plot results
        evaluator.plot_heatmap(ig_attr, f"Sample {i+1} - Integrated Gradients")
        evaluator.plot_summary(ig_attr, f"Sample {i+1} - Integrated Gradients Summary")
        
        evaluator.plot_heatmap(sal_attr, f"Sample {i+1} - Saliency")
        evaluator.plot_summary(sal_attr, f"Sample {i+1} - Saliency Summary")

quick_evaluation(model, test_loader, config, device, num_samples=3)